**Generative AI Use**: For the purposes of the assignments, the use of generative AI is subject to the same policies regarding collaboration. Just as with other collaborators, each student must write down the solutions independently of the output of the interaction and the submission should include a note denoting the nature of the collaboration. The use of generative AI tools to substantially complete sections of the assignments is not in line with the spirit of the assignments, and would be a violation of the [Honor Code](https://communitystandards.stanford.edu/policies-and-guidance/honor-code).


---

**中文对照（参考译文，不含解答）**

**生成式 AI 的使用**：本作业对生成式 AI 的使用采用与协作相同的规则。与其他协作者交流之后，每位学生仍须独立写出自己的解答，不能直接依赖交流中得到的输出；提交时应附上说明，如实记录协作的性质。使用生成式 AI 实质性地完成作业的大部分内容，不符合本作业的教学目的，也会违反[学术诚信准则](https://communitystandards.stanford.edu/policies-and-guidance/honor-code)。

In [ ]:
# This mounts your Google Drive to the Colab VM.
# [中文] 这把Google车开到Colab VM
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# TODO: Enter the foldername in your Drive where you have saved the unzipped
# assignment folder, e.g. 'cs231n/assignments/assignment3/'
# [中文] TODO: 在您的驱动器中输入文件夹名, 在那里您保存了未覆盖的任务文件夹e.g。 “ cs231n/ assignations/
# formantions3/ ”
FOLDERNAME = "cs231n/assignments/assignment3/"
assert FOLDERNAME is not None, "[!] Enter the foldername."

# Now that we've mounted your Drive, this ensures that
# the Python interpreter of the Colab VM can load
# python files from within it.
# [中文] 现在,我们已经安装了你的驱动器, 这确保了Python 翻译 科拉布VM 可以装载
import sys
sys.path.append('/content/drive/My Drive/{}'.format(FOLDERNAME))

In [ ]:
# This downloads the COCO dataset to your Drive if it doesn't already exist
# (you should already have this dataset from a previous notebook!)
# Uncomment the following if you don't have it.
# %cd /content/drive/My\ Drive/$FOLDERNAME/cs231n/datasets/
# !bash get_coco_captioning.sh
# %cd /content/drive/My\ Drive/$FOLDERNAME
# [中文] 如果您的驱动器不存在的话, 此选项会下载 COCO 数据集( 您应该已经从上一个笔记本上下载此数据集 )。 如果您没有的话, 请取消以下的注释。


In [ ]:
# Some useful python libraries
# [中文] 一些有用的 python 库
! pip install ftfy regex tqdm
! pip install git+https://github.com/openai/CLIP.git
! pip install decord

# State-of-the-Art Pretrained Image Models

In the previous exercise, you learned about [SimCLR](https://arxiv.org/abs/2002.05709) and how contrastive self-supervised learning can be used to learn meaningful image representations. In this notebook, we will explore two more recent models that also aim to learn high-quality visual representations and have demonstrated strong and robust performance on a variety of downstream tasks.


First, we will examine the [CLIP](https://github.com/openai/CLIP) model. Like SimCLR, CLIP uses a contrastive learning objective, but instead of contrasting two augmented views of the same image, it contrasts two different modalities: text and image. To train CLIP, OpenAI collected a large dataset of ~400M image-text pairs from the internet, including sources like Wikipedia and image alt text. The resulting model learns rich, high-level image features and has achieved impressive zero-shot performance on many vision benchmarks.

Next, we will explore [DINO](https://github.com/facebookresearch/dino), a self-supervised learning method for vision tasks that applies contrastive learning in a self-distillation framework with multi-crop augmentation strategy. The authors showed that the features learned by DINO ViTs are fine-grained and semantically rich with explicit information about the semantic segmentation of the image.




---

**中文对照（参考译文，不含解答）**

# 最先进的预设图像模型

在前一次练习中,您了解了 [SimCLR](https://arxiv.org/abs/2002.05709) 和如何利用对比性自我监督学习来学习有意义的图像表达方式。 在这个笔记本中,我们将探索另外两个最新模型,这些模型还旨在学习高质量的视觉表达方式,并在各种下游任务中表现出强劲有力的表现。


首先, 我们将检查 [CLIP](https://github.com/openai/CLIP) 模型。 与 SimCLR 一样, CLIP 使用一个对比性学习目标, 而不是对比同一图像的两种扩大视图, 它对比了两种不同的模型: 文本和图像。 为了训练 CLIP, OpenAI 从互联网上收集了一大套 ~ 400M 图像- 文本, 包括维基百科 和 图像变异文本等源。 由此生成的模型学习了丰富、 高层次的图像特征, 并且在许多愿景基准上取得了令人印象深刻的零效果。

接下来,我们将探索[DINO](https://github.com/facebookresearch/dino)这是一种自我监督的视觉任务学习方法,在自我提炼框架中应用对比性学习,并采用多作物增殖战略。 作者们显示,DINO ViTs所学的特征精细精细,在语义上丰富,有清晰的图像语义分解信息。




# CLIP

As explained above, CLIP's training objective incorporates both text and images, building upon the principles of contrastive learning. Consider this quote from the SimCLR notebook:
>The goal of the contrastive loss is to maximize agreement between the final vectors **$z_i = g(h_i)$** and **$z_j = g(h_j)$**.

Similarly, CLIP is trained to maximize agreement between two vectors. However, because these vectors come from different modalities, CLIP uses two separate encoders: a transformer-based Text Encoder and a Vision Transformer (ViT)-based Image Encoder. Note that some smaller, more efficient versions of CLIP use a ResNet as the Image Encoder instead of a ViT.

Run the cell below to visualize the training and inference pipeline of CLIP.

During the pretraining phase, each batch consists of multiple images along with their corresponding captions. Each image is independently processed by an Image Encoder—typically a visual model like a Vision Transformer (ViT) or a Convolutional Neural Network (ConvNet)—which produces an image embedding $I_n$. Likewise, each caption is independently processed by a Text Encoder to generate a corresponding text embedding $T_n$. Next, we compute the pairwise similarities between all image-text combinations, meaning each image is compared with every caption, and vice versa. The training objective is to maximize the similarity scores along the diagonal of the resulting similarity matrix -- that is, the scores for the matching image-caption pairs $(I_n, T_n)$.  Through backpropagation, the model learns to assign higher similarity scores to true matches than to mismatched pairs.

Through this setup, CLIP effectively learns to represent images and texts in a shared latent space. In this space, semantic concepts are encoded in a modality-independent way, enabling meaningful cross-modal comparisons between visual and textual inputs.



---

**中文对照（参考译文，不含解答）**

# CLIP

如上所述,CLIP的训练目标既包括文字,也包括图像,其基础是对比性学习的原则。
>对比性损失的目标是使最终矢量**$z_i = g(h_i)$**和**$z_j = g(h_j)$**之间达成最大程度的协议。

同样,CLIP也受过训练,以最大限度地实现两个矢量之间的一致。然而,由于这些矢量来自不同的模式,CLIP使用两个不同的编码器:一个基于Transformer的文本编码器和一个基于图像编码器的愿景转换器(ViT)。请注意,一些较小的、更有效的CLIP版本使用ResNet作为图像编码器,而不是维特。

运行下面的单元 想象CLIP的训练和推论管道。

在训练前阶段,每批图像由多个图像及其相应描述组成。 每组图像由图像编码器独立处理, 通常由视觉转换器(ViT) 或卷积神经网络(ConvNet) 等视觉模型独立处理, 它产生图像嵌入 $I_n$。 同样, 每个描述由文本编码器独立地处理, 产生相应的文本嵌入 $T_n$。 接下来, 我们计算所有图像文本组合的对等相似性, 意即每个图像与每个描述比较, 反之亦然。 训练的目的是尽量扩大相近性分数沿图像-。 生成的相似矩阵的对角值 -- -- 即匹配图像插图配对的分数 $(I_n, T_n)$。 通过 反向传播, 模型学会给真实匹配的相近分数比不匹配的对匹配的分数高。

通过这个设置, CLIP 有效地学会在共享的潜在空间中代表图像和文本。 在这个空间中,语义概念以独立于模式的方式编码,从而能够对视觉输入和文字输入进行有意义的交叉模式比较。



In [ ]:
from IPython.display import Image as ColabImage
ColabImage(f'/content/drive/My Drive/{FOLDERNAME}/CLIP.png')

**Inline Question 1** -

Why does CLIP's learning depend on the batch size? If the batch size is fixed, what strategy can we use to learn rich image features?

$\color{blue}{\textit Your Answer:}$




---

**中文对照（参考译文，不含解答）**

**随堂问题 1**

为什么 CLIP 的学习效果依赖批大小？如果批大小固定，可以采用什么策略来学习丰富的图像特征？

**你的回答：** [请自行填写]

# Loading COCO dataset

We'll use the same captioning dataset you used to train your RNN captioning model, but instead of generating the captions lets see if we can match each image to the correct caption.

---

**中文对照（参考译文，不含解答）**

# 正在装入COCO数据集

我们将使用您用来训练 RNN 字幕模型的同一字幕数据集, 而不是生成字幕, 让我们看看我们是否能将每个图像匹配到正确的字幕。

In [ ]:
import sys
import types
import importlib

if "imp" not in sys.modules:
    imp = types.ModuleType("imp")
    imp.reload = importlib.reload
    sys.modules["imp"] = imp

%load_ext autoreload
%autoreload 2

import time, os, json
import numpy as np
import matplotlib.pyplot as plt
import torch
import clip
import torch
from tqdm.auto import tqdm

from PIL import Image
from cs231n.clip_dino import *

def rel_error(x, y):
    """Returns relative error."""
    return np.max(np.abs(x - y) / (np.maximum(1e-10, np.abs(x) + np.abs(y))))


In [ ]:
from cs231n.coco_utils import load_coco_data, sample_coco_minibatch, decode_captions
from cs231n.image_utils import image_from_url

In [ ]:
# Load COCO data from disk into a dictionary.
# this is the same dataset you used for the RNN captioning notebook :)
# [中文] 将 COCO 数据从磁盘装入字典。 这是 RNN 描述笔记本所用的相同数据集:
data = load_coco_data(pca_features=True)

# Print out all the keys and values from the data dictionary.
# [中文] 从数据字典中打印所有键和值。
for k, v in data.items():
    if type(v) == np.ndarray:
        print(k, type(v), v.shape, v.dtype)
    else:
        print(k, type(v), len(v))

In [ ]:
# we're just using the loaded captions from COCO, so we need to decode them and get rid of the special tokens.
# [中文] 我们只是使用COCO上载的字幕 所以我们需要解码 并摆脱特殊标志。
decoded_captions= []
for caption in data['val_captions']:
  caption = decode_captions(caption, data['idx_to_word'])\
    .replace('<START>', '')\
    .replace('<END>', '')\
    .replace('<UNK>', '')\
    .strip()
  decoded_captions.append(caption)

In [ ]:
# lets get 10 examples
# [中文] 让我们获取 10 示例
mask = np.array([135428, 122586, 122814, 133173, 176639, 163828,  98169,   6931,
        19488, 175760])
first_captions = [decoded_captions[elem] for elem in mask]

img_idxs = data['val_image_idxs'][mask]       # the images the captions refer to
# [中文] 描述所指的图像
first_images   = [image_from_url(data['val_urls'][j]) for j in img_idxs]

In [ ]:
for i, (caption, image) in enumerate(zip(first_captions, first_images)):
    plt.imshow(image)
    plt.axis('off')
    caption_str = caption
    plt.title(caption_str)
    plt.show()

# Running the CLIP Model

First we'll use the pretrained CLIP model to extract features from the texts and images separetely.

---

**中文对照（参考译文，不含解答）**

# 运行 CLIP 模型

首先,我们将使用预先训练的CLIP模型 从文本和图像中分离出特性。

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
clip_model, clip_preprocess = clip.load("ViT-B/32", device=device)

In [ ]:
# You can check the model layers by printing the model.
# CLIP's model code is available at https://github.com/openai/CLIP/tree/main/clip
# print(clip_model)
# [中文] 您可以通过打印模型来检查模型层。 CLIP 的模型代码可在 https://github.com/openai/CLIP/tree/main/clip
# 上查阅


In [ ]:
# First, we encode the captions into vectors in the shared embedding space.
# Since we're using a Transformer as the text encoder, we need to tokenize the text first.
# [中文] 首先,我们将描述编码为共享嵌入空间中的矢量。由于我们正在使用变换器作为文本编码器,所以我们需要先将文本标记下来。
text_tokens = clip.tokenize(first_captions).to(device)
with torch.no_grad():
    text_features = clip_model.encode_text(text_tokens)

# Sanity check, print the shape
# [中文] 检查是否干净,打印形状
print(text_features.shape)

# Note: For your implementations, use the above clip_model.encode_text() function. Avoid using clip_model.forward().
# [中文] 注意: 对于您的执行, 请使用上述 clip_model.encode_text() 函数。 避免使用 clip_model.forward()。


In [ ]:
# Then, we encode the images into the same embedding space.
# [中文] 然后,我们把图像编码成相同的嵌入空间。
processed_images = [
    clip_preprocess(Image.fromarray(img)).unsqueeze(0)
    for img in first_images
]
images_tensor = torch.cat(processed_images, dim=0).to(device)

with torch.no_grad():
    image_features = clip_model.encode_image(images_tensor)

# sanity check, print the shape
# [中文] 理智检查,打印形状
print(image_features.shape)

# Note: For your implementations, use the above clip_model.encode_image() function. Avoid using clip_model.forward().
# [中文] 注意: 对于您的执行, 请使用上述 clip_model.encode_image() 函数。 避免使用 clip_model.forward()。


Open `cs231n/clip_dino.py` and implement `get_similarity_no_loop` to compute similarity scores between text features and image features. Test your implementation below, you should see relative errors less than 1e-5.

---

**中文对照（参考译文，不含解答）**

打开 `cs231n/clip_dino.py` 并安装 `get_similarity_no_loop` 以计算文本特性和图像特性之间的相似分数。 在下面测试您的执行, 您应该看到比 1e-5 更少的相对错误。

In [ ]:
from cs231n.clip_dino import get_similarity_no_loop
torch.manual_seed(231)
np.random.seed(231)
M, N, D = 5, 6, 10

test_text_features = torch.randn(N, D)
test_image_features = torch.randn(M, D)
out = get_similarity_no_loop(test_text_features, test_image_features)

expected_out = np.array([
    [ 0.1867811 , -0.23494351,  0.44155994, -0.18950461,  0.00100103],
    [ 0.17905031, -0.25469488, -0.64330417,  0.25097957, -0.09327742],
    [-0.4407011 , -0.4365381 ,  0.32857686, -0.3765278 ,  0.01049389],
    [ 0.24815483,  0.42157224, -0.08459304,  0.14132318, -0.26935193],
    [ 0.02309848, -0.01441101,  0.5469337 ,  0.6018773 ,  0.21581158],
    [ 0.41579214, -0.014449  , -0.7242257 ,  0.39348006,  0.0822239 ],
]).astype(np.float32)

print("relative error: ", rel_error(out.numpy(), expected_out))

In [ ]:
# Let's visualize the similarities between our batch of images and their captions.
# [中文] 让我们想象一下我们系列的图像 和它们的字幕之间的相似之处。

similarities = get_similarity_no_loop(text_features, image_features).cpu().detach().numpy()

plt.figure(figsize=(20, 14))
plt.imshow(similarities, vmin=0.1, vmax=0.3)
plt.yticks(range(len(text_features)), first_captions, fontsize=18)
plt.xticks([])
for i, image in enumerate(first_images):
    plt.imshow(image, extent=(i - 0.5, i + 0.5, -1.6, -0.6), origin="lower")
for x in range(similarities.shape[1]):
    for y in range(similarities.shape[0]):
        plt.text(x, y, f"{similarities[y, x]:.2f}", ha="center", va="center", size=12)

for side in ["left", "top", "right", "bottom"]:
    plt.gca().spines[side].set_visible(False)

plt.xlim([-0.5, len(image_features) - 0.5])
plt.ylim([len(text_features) + 0.5, -2])

plt.title("Cosine similarity between text and image features", size=20)
plt.show()

# Zero Shot Classifier

You will be able to see a high similarity between matching image-caption pairs above. We can leverage this property to design an image classifier that doesn't require any labeled data (i.e., a zero-shot classifier). Each class can be represented using an appropriate natural language description, and any input image will be classified into the class whose description has the highest similarity with the image in CLIP's embedding space.

---

**中文对照（参考译文，不含解答）**

# 零射击分类器

您可以看到上面匹配的图像插图配对之间的高度相似性。 我们可以利用此属性来设计一个不需要任何标签数据的图像分类器( i.e ., 零弹分解器 )。 每个类都可以使用适当的自然语言描述来表示, 任何输入图像都将被分类为与 CLIP 嵌入空间中图像最相似的类。

Implement `clip_zero_shot_classifier` in `cs231n/clip_dino.py` and test it below. You should be able to see the following predictions:

['a person', 'an animal', 'an animal', 'food', 'a person', 'a landscape', 'other', 'other', 'other', 'a person']

---

**中文对照（参考译文，不含解答）**

在 `cs231n/clip_dino.py` 中执行 `clip_zero_shot_classifier` 并在下面测试它。 您应该能看到以下预测:

[人,动物,动物],'food'"一个人","风景",'other',  'other',  'other',"一个人"

In [ ]:
from cs231n.clip_dino import clip_zero_shot_classifier

classes = ["a person", "an animal", "food", "a landscape", "other"]

pred_classes = clip_zero_shot_classifier(
    clip_model, clip_preprocess, first_images, classes, device)

print(pred_classes)

Run the cell below to visualize the predictions. As you can see, CLIP offers a straightforward way to perform reasonable zero-shot classification across any class taxonomy.

CLIP was the first model to outperform standard supervised training on ImageNet classification without using any ImageNet images or labels (The original CLIP paper has many such interesting experiments and analysis).


---

**中文对照（参考译文，不含解答）**

运行下面的单元格来直观预测。 正如您所看到的, CLIP 提供了在任何分类中进行合理的零射分类的直截了当的方法。

CLIP是第一个在不使用任何图像网络图像或标签(The original CLIP paper has many such interesting experiments and analysis)的情况下完成图像网络分类标准监督训练的模型。


In [ ]:
# Visualize the zero shot predictions
# [中文] 视觉化零射线预测
for i, (pred_class, image) in enumerate(zip(pred_classes, first_images)):
    plt.imshow(image)
    plt.axis('off')
    plt.title(pred_class)
    plt.show()

# Image Retrieval using CLIP

Just as we used CLIP to retrieve the matching class name for each image, we can also use it to retrieve matching images from text inputs (semantic image retrieval). Implement the `CLIPImageRetriever` in `cs231n/clip_dino.py` and test it by running the two cells below. The expected top 2 outputs for each query are provided in the comments.

---

**中文对照（参考译文，不含解答）**

# 使用 CLIP 获取图像

正如我们使用 CLIP 获取每个图像的匹配类名称一样, 我们也可以使用它从文本输入 (semantic image retrieval) 中获取匹配的图像。 在 `cs231n/clip_dino.py` 中执行 `CLIPImageRetriever`, 并通过运行下面的两个单元格来测试它。 每个查询的预期顶端 2 输出在注释中提供。

In [ ]:
from cs231n.clip_dino import CLIPImageRetriever
clip_retriever = CLIPImageRetriever(clip_model, clip_preprocess, first_images, device)

In [ ]:
query = "sports"  # tennis, skateboard
# query = "black and white"  # bathroom, zerbas
# [中文] 网球,滑板
img_indices = clip_retriever.retrieve(query)

for img_index in img_indices:
    plt.imshow(first_images[img_index])
    plt.axis('off')
    plt.show()

**Inline Question 2** -

CLIP learns to align image and text representations in a shared latent space using a contrastive loss. How would you extend this idea to more than two modalities?

$\color{blue}{\textit Your Answer:}$



---

**中文对照（参考译文，不含解答）**

**随堂问题 2**

CLIP 使用对比损失，将图像表示与文本表示对齐到一个共享的潜在空间。如何把这一思路扩展到两个以上的模态？

**你的回答：** [请自行填写]

# DINO

As mentioned earlier, models trained with vanilla contrastive learning methods such as SimCLR and CLIP require very large batch sizes. This makes them computationally expensive and limits their accessibility. Subsequent works, like [BYOL](https://arxiv.org/abs/2006.07733), propose an alternative approach that avoids the need for numerous negative samples by using a student-teacher framework. This method performs surprisingly well and was later adopted by [DINO](https://arxiv.org/abs/2104.14294) .

Similar to SimCLR, DINO is trained to maximize the agreement between two vectors derived from different views of the same image. However, unlike SimCLR, DINO uses two separate encoders which are trained differently. The student network is updated via backpropagation to match the outputs of the teacher network. The teacher network is not updated via backpropagation; instead, its weights are updated using an exponential moving average (EMA) of the student's weights. This means that the teacher model evolves more slowly and provides a stable target for the student to learn from.

Run the cell below to visualize the DINO training pipeline.

---

**中文对照（参考译文，不含解答）**

# DINO

如前所述,使用香草对比学习方法(如SimCLR和CLIP)训练的模型需要非常大的批量尺寸,这使得它们计算成本昂贵,限制了它们的可及性。随后的作品,如[BYOL](https://arxiv.org/abs/2006.07733),提出了一种替代方法,通过使用学生-教师框架,避免大量负面样本。这种方法效果惊人,后来被[DINO](https://arxiv.org/abs/2104.14294)采用。

与 SimCLR 相似, DINO 也接受过训练, 以最大限度地实现来自同一图像不同观点的两种矢量之间的一致。 但是, 与 SIMCLR不同的是, DINO 使用两种不同的编码器, 其训练方式不同。 学生网络通过 反向传播 进行更新, 以匹配教师网络的输出。 教师网络没有通过 反向传播 更新; 相反, 其权重是使用学生体重的指数移动平均值 (EMA) 来更新的。 这意味着教师模型的进化速度更慢, 为学生学习提供了一个稳定的目标。

运行下面的单元 视觉化DINO训练管道。

In [ ]:
from IPython.display import Image as ColabImage
ColabImage(f'/content/drive/My Drive/{FOLDERNAME}/dino.gif')

In [ ]:
# first let's get rid of the CLIP model that's currently using memory
# [中文] 先把目前使用内存的CLIP模型除掉
del clip_model
# Uncomment the following if you are using GPU runtime
# torch.cuda.empty_cache()
# torch.cuda.ipc_collect()
# [中文] 如果您使用GPU 运行时间, 取消下列注释


In [ ]:
# Load smallest dino model. ViT-S/8. Here ViT-S has ~22M parameters and
# works on 8x8 patches.
# [中文] 装入最小的 dino 模型。 Vit- S / 8。 VIT- S 有 ~ 22M 参数, 在 8x8 图像块上工作。
dino_model = torch.hub.load('facebookresearch/dino:main', 'dino_vits8')
dino_model.eval().to(device)


In [ ]:
# the image we will be playing around with
# [中文] 我们将会玩弄的画面
sample_image = Image.fromarray(first_images[0]).convert("RGB")
sample_image

# DINO Attention Maps

Since the loaded DINO checkpoint is based on the ViT architecture, we can visualize what each attention head is focusing on. The code below generates heatmaps showing which patches of the original image the [CLS] token attends to across the various heads in the final layer. Although this model was trained using a self-supervised objective without any explicit instruction to recognize "structure" in images, still...

Do you notice any patterns?

---

**中文对照（参考译文，不含解答）**

# DINO 关注地图

由于装入的 DINO 检查站以 VIT 结构为基础, 我们可以直观地看到每个关注头的焦点。 下面的代码生成热图, 显示原始图像的哪些图像块, 在最后一层, [CLS] 符号会覆盖各个头部。 虽然这个模型是用一个自我监督的目标来训练的, 但没有明确的指令来识别图像中的“ 结构 ”, 但是...

你注意到任何模式吗?

In [ ]:
# Preprocess
# [中文] 预处理
from torchvision import transforms as T
transform = T.Compose([
    T.Resize((480, 480)),
    T.ToTensor(),
    T.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])
img_tensor = transform(sample_image)
w, h = img_tensor.shape[1:]
img_tensor = img_tensor[None].to(device)

# Extract attention
# [中文] 解开关注
with torch.no_grad():
    attn = dino_model.get_last_selfattention(img_tensor)[0, :, 0, 1:]
nh, tokens = attn.shape
w_feat, h_feat = w // 8, h // 8
attn = attn.reshape(nh, w_feat, h_feat)
attn = torch.nn.functional.interpolate(attn.unsqueeze(0), scale_factor=8, mode="nearest")[0].cpu().numpy()

# Plot attention heads
# [中文] 绘制关注头
fig, axes = plt.subplots(1, nh, figsize=(3 * nh, 3))
for i in range(nh):
    ax = axes[i] if nh > 1 else axes
    ax.imshow(attn[i], cmap='inferno')
    ax.axis('off')
plt.show()

In [ ]:
# Extract patch token features and discard [CLS] token.
# [中文] 提取图像块符号特性和丢弃标记。
with torch.no_grad():
    all_tokens = dino_model.get_intermediate_layers(img_tensor, n=1)[0]  # (1, 1+N, D)
    patch_tokens = all_tokens[:, 1:, :]  # (N, D)

print(img_tensor.shape)
print(all_tokens.shape)
print(patch_tokens.shape)


**Inline Question 3**

How do we get the tensor shapes printed above? Explain your answer.


$\color{blue}{\textit Your Answer:}$






---

**中文对照（参考译文，不含解答）**

**随堂问题 3**

上方打印出的张量形状是怎样得到的？请解释你的答案。

**你的回答：** [请自行填写]

# DINO Features

To understand what the model is encoding in each patch, we can visualize the contents of each patch token. Since these embeddings are high-dimensional and difficult to interpret directly, we'll use PCA to identify the directions of highest variance in the feature space.

In the next cell, we visualize the three principal directions of variance in the feature space. This reveals the dominant structure that the patch embeddings are capturing.

---

**中文对照（参考译文，不含解答）**

# DINO 特性

要理解每个图像块的编码模型, 我们可以看到每个图像块标记的内容。 由于这些嵌入是高维且难以直接解释的, 我们将使用 主成分分析（PCA） 来识别特性空间的最大差异方向。

在下一个单元格中,我们可以想象地物空间的三个主要差异方向。这显示了图像块嵌入所捕捉的主导结构。

In [ ]:
from sklearn.decomposition import PCA

np.random.seed(231)

# PCA
# [中文]  主成分分析（PCA）
pca = PCA(n_components=3)
patch_pca = pca.fit_transform(patch_tokens.cpu().numpy()[0])

# Normalize PCA components to [0, 1] for RGB display
# [中文] 将 RGB 显示的 主成分分析（PCA） 组件正常化到 [0, 1]
patch_rgb = (patch_pca - patch_pca.min(0)) / (patch_pca.max(0) - patch_pca.min(0))

# Reshape to image grid (60x60, 3)
# [中文] 重塑为图像网格 (60x60, 3)
patch_rgb_img = patch_rgb.reshape(60, 60, 3)

# Show as image
# [中文] 显示为图像
plt.figure(figsize=(6, 6))
plt.imshow(patch_rgb_img)
plt.axis('off')
plt.title("Patch Embeddings (PCA → RGB)")
plt.show()

**Inline Question 4** -

What kind of structure do you see in the visualization above? What does it imply when a region consistently appears in a specific color? What does it mean when two regions have distinctly different color? Remember that PCA reveals the directions of highest variance in the feature space across all patches. A patch's color reflects its distinct feature content.


$\color{blue}{\textit Your Answer:}$




---

**中文对照（参考译文，不含解答）**

**随堂问题 4**

在上方的可视化结果中，你看到了什么结构？如果某一区域始终呈现特定颜色，意味着什么？如果两个区域的颜色明显不同，又意味着什么？

请记住：PCA 揭示的是所有图像块在特征空间中方差最大的方向。某个图像块的颜色反映了它所包含的特征内容。

**你的回答：** [请自行填写]

# A Simple Segmentation Model over DINO Features

In the previous section, we saw that DINO features can provide surprisingly good segmentation cues. Now, let's put that idea to the test by training a simple segmentation model on the [DAVIS dataset](https://davischallenge.org). The DAVIS dataset (Densely Annotated VIdeo Segmentation) was created for video object segmentation tasks. It provides frame-by-frame, pixel-level annotations of objects within videos. For this experiment, we'll train our model using the annotations from just a single frame of a video and see how well it performs on the remaining frames of the same.

Our model will be intentionally minimal: we'll extract DINO features per patch and train a lightweight per-patch classifier using only the patches from that one annotated frame. Typically, you would train on the full dataset and evaluate on a separate validation set containing different videos. But here, we will test the one-shot capabilities of DINO features.







---

**中文对照（参考译文，不含解答）**

# DINO 特性上的简单分区模型

在上一节中,我们看到 DINO 特性可以提供出人意料的良好分解提示。 现在, 让我们通过在 [DAVIS dataset](https://davischallenge.org) 上训练一个简单的分解模型来测试这个想法。 DAVIS 数据集 (Densely Annotated VIdeo Segmentation) 是为视频对象分解任务创建的。 它提供了视频中对象的边框、 像素级的注释。 对于这个实验, 我们将使用一个视频的单个框来训练我们的模型, 并查看它是如何在剩余框中运行的。

我们的模型将刻意最小化: 我们将提取每个图像块的 DINO 特性, 并训练一个轻量级的 / per patch 分类器, 仅使用该附加注释的框中的图像块。 通常, 您会用完整的数据集来训练, 并对包含不同视频的单独验证集进行评估。 但在这里, 我们将测试 DINO 各特性的一分能力。







In [ ]:
from cs231n.clip_dino import DavisDataset

# A helper class to work with DAVIS dataset.
# It may take ~5 minutes on the first run of this cell to download the dataset.
# [中文] 使用 DAVIS 数据集的辅助类。 此单元格第一次运行时可能需要 ~ 5 分钟才能下载数据集。
davis_ds = DavisDataset()

# Get a specific test video. Do NOT change this for submission.
# [中文] 获取一个特定的测试视频。 不要更改此选项以提交。
frames, masks = davis_ds.get_sample(7)
num_classes = masks.max() + 1

print(frames.shape, masks.shape, num_classes)

In [ ]:
# Get DINO patch features and corresponding class labels for a middle frame
# [中文] 获取中间框的 DINO 图像块和相应的类标签
train_fi = 40
X_train = davis_ds.process_frames(frames[train_fi:train_fi+1], dino_model, device)[0]
Y_train = davis_ds.process_masks(masks[train_fi:train_fi+1], device)[0]
print(X_train.shape, Y_train.shape)

Complete the implementation of the `DINOSegmentation` class in `cs231n/clip_dino.py`, and test it by running the two cells below. You should achieve a mean IoU greater than 0.45 on the first test frame and greater than 0.50 on the last test frame. To prevent overfitting on the training patch features, consider designing a very lightweight model (e.g., a linear layer or a 2-layer MLP) and applying appropriate weight decay.

You may use GPU runtime to speed up training and evaluation. Make sure to rerun the entire notebook if you change runtime type.

---

**中文对照（参考译文，不含解答）**

完成 `DINOSegmentation` 类在 `cs231n/clip_dino.py` 中的执行, 并通过运行下面的两个单元格来测试它。 您应该达到第一个测试框上大于 0.45 且在最后一个测试框上高于 0.50 的平均IoU。 为了防止在训练图像块特征上过度适应, 请考虑设计一个非常轻的模型( e.g 、 线性层或 2 - 层的 MLP), 并应用适当的权重衰减。

您可以使用 GPU 运行时间来加速训练和评价。 如果您更改运行时间类型, 请确保重运行整个笔记本。

In [ ]:
from cs231n.clip_dino import DINOSegmentation, compute_iou
torch.manual_seed(231)
np.random.seed(231)
dino_segmentation = DINOSegmentation(device, num_classes)
dino_segmentation.train(X_train, Y_train, num_iters=500)


# Test on first, middle, and last frame
# [中文] 测试第一、中、最后一个框架
ious = []
test_fis = [0, train_fi, 98]
gt_viz = []
pred_viz = []
for fi in test_fis:
  X_test = davis_ds.process_frames(frames[fi:fi+1], dino_model, device)[0]
  Y_test = davis_ds.process_masks(masks[fi:fi+1], device)[0]
  Y_pred = dino_segmentation.inference(X_test)
  iou = compute_iou(Y_pred, Y_test, num_classes)
  ious.append(iou)

  gt_viz.append(davis_ds.mask_frame_overlay(Y_test, frames[fi]))
  pred_viz.append(davis_ds.mask_frame_overlay(Y_pred, frames[fi]))

gt_viz = np.concatenate(gt_viz, 1)
pred_viz = np.concatenate(pred_viz, 1)

In [ ]:
print(f"Mean IoU on first test frames: {ious[0]:.3f}")  # should be >0.45
# [中文] 应该是 > 0.45
print(f"Mean IoU on last test frames: {ious[2]:.3f}")  # should be >0.50
# [中文] 应该是 > 0.50


Now let's visualize the results. Run the two cells below to display the ground truth and predicted segmentation masks for the first, middle, and last frames. Note that the middle frame is part of the training set, while the other frames are unseen.

---

**中文对照（参考译文，不含解答）**

现在让我们想象一下结果。 运行下面的两个单元格, 以显示第一个、 中间 和最后一个框架的地面真相和预测分割面罩。 注意中间框是训练组合的一部分, 而其他框架是看不见的。

In [ ]:
Image.fromarray(gt_viz)

In [ ]:
Image.fromarray(pred_viz)

Now run the following three cells to evaluate and visualize the entire video. You should achieve a mean IoU greater than 0.55. The saved visualization video may take some time to process in Google Drive, but you can download it to your computer and view it locally.



---

**中文对照（参考译文，不含解答）**

现在运行以下三个单元格来评估和视觉化整个视频。 您应该实现一个比 0.55 更大的平均 IOU。 在 Google 驱动器中保存的可视化视频可能需要一些时间处理, 但是您可以下载到您的计算机上并在当地查看它。



In [ ]:
# Run on all frames
# [中文] 在所有框架上运行
ious = []
gt_viz = []
pred_viz = []
for fi in range(len(frames)):
  if fi % 20 == 0:
    print(f"{fi} / {len(frames)}")
  X_test = davis_ds.process_frames(frames[fi:fi+1], dino_model, device)[0]
  Y_test = davis_ds.process_masks(masks[fi:fi+1], device)[0]
  Y_pred = dino_segmentation.inference(X_test)
  iou = compute_iou(Y_pred, Y_test, num_classes)
  ious.append(iou)

  gt_viz.append(davis_ds.mask_frame_overlay(Y_test, frames[fi]))
  pred_viz.append(davis_ds.mask_frame_overlay(Y_pred, frames[fi]))

gt_viz = np.stack(gt_viz)  # T x H x W x 3
pred_viz = np.stack(pred_viz)  # T x H x W x 3
final_viz = np.concatenate([gt_viz, pred_viz], -2)  # T x H x 2W x 3

In [ ]:
print(f"Mean IoU on all frames: {sum(ious) / len(ious):.3f}")  # should be >0.55
# [中文] 应该是 > 0.55


In [ ]:
def write_video_from_array(array, output_path, fps = 12):
    T, H, W, _ = array.shape
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (W, H))
    for i in range(T):
        frame = array[i]
        out.write(frame)
    out.release()
    print(f"Video saved to {output_path}")


# It might take a while to process in google drive but you can just download it and watch on your computer
# [中文] 在谷歌驱动器中处理可能需要一段时间,但你可以下载它,在电脑上观看
write_video_from_array(final_viz, f"/content/drive/My Drive/{FOLDERNAME}/dino_res.mp4")

**Inline Question 5** -

If you train a segmentation model on CLIP ViT's patch features, do you expect it to perform better or worse than DINO? Why should that be the case?



$\color{blue}{\textit Your Answer:}$




---

**中文对照（参考译文，不含解答）**

**随堂问题 5**

如果在 CLIP ViT 的图像块特征上训练分割模型，你预期其效果会优于还是弱于 DINO？为什么？

**你的回答：** [请自行填写]